In [2]:
from pyspark.sql import SparkSession

### Answer No1

In [6]:
spark = SparkSession.builder\
        .master("local[*]")\
        .appName('test')\
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/07 12:53:03 WARN Utils: Your hostname, DESKTOP-LQFDL79, resolves to a loopback address: 127.0.1.1; using 172.22.154.148 instead (on interface eth0)
26/03/07 12:53:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 12:53:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
spark.version

'4.1.1'

In [8]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')

### Answer No2

In [10]:
df_yellow.repartition(4).write.parquet('pq/')

### Answer No3

In [12]:
df_yellow.registerTempTable('taxi_data')

In [21]:
df_yellow_sel = spark.sql("""
SELECT
    COUNT(*) AS total_trip
FROM
    taxi_data
where Date(tpep_pickup_datetime) = Date('2025-11-15')
""").show()

+----------+
|total_trip|
+----------+
|    162604|
+----------+



### Answer No4

In [38]:
df_yellow_distance = spark.sql("""
SELECT
    MAX(
        (unix_timestamp(tpep_dropoff_datetime) - 
         unix_timestamp(tpep_pickup_datetime)) / 3600
    ) AS longest_trip_hours
FROM
    taxi_data
""").show()

+------------------+
|longest_trip_hours|
+------------------+
| 90.64666666666666|
+------------------+



### Answer No6

In [41]:
df_zones = spark.read\
            .option('header','true')\
            .csv('taxi_zone_lookup.csv')

In [45]:
df_zones.registerTempTable('taxi_zones')

In [55]:
df_sel = spark.sql("""
SELECT
    PULocationID,
    L.Zone,
    COUNT(PULocationID) AS total
FROM 
    taxi_data
JOIN taxi_zones L ON taxi_data.PULocationID = L.LocationID
GROUP BY 1,2
ORDER BY 3 ASC
""").show()

[Stage 41:=============================>                            (1 + 1) / 2]

+------------+--------------------+-----+
|PULocationID|                Zone|total|
+------------+--------------------+-----+
|         105|Governor's Island...|    1|
|          84|Eltingville/Annad...|    1|
|           5|       Arden Heights|    1|
|         187|       Port Richmond|    3|
|         204|   Rossville/Woodrow|    4|
|         199|       Rikers Island|    4|
|         111| Green-Wood Cemetery|    4|
|         109|         Great Kills|    4|
|           2|         Jamaica Bay|    5|
|         251|         Westerleigh|   12|
|         176|             Oakwood|   14|
|         172|New Dorp/Midland ...|   14|
|          59|        Crotona Park|   14|
|         245|       West Brighton|   14|
|         253|       Willets Point|   15|
|          27|Breezy Point/Fort...|   16|
|         206|Saint George/New ...|   17|
|          30|       Broad Channel|   18|
|         156|     Mariners Harbor|   21|
|         118|Heartland Village...|   22|
+------------+--------------------